# 02 — Model Training & Evaluation

**PathSnap GPS Map-Matching | PES University UE24CS352A**

This notebook:
1. Generates a synthetic dataset that mirrors the real feature distribution (so the notebook runs stand-alone without data)
2. Applies `StandardScaler` and handles class imbalance
3. Trains **7 classifier families** with 5-fold stratified CV
4. Produces comparison tables, bar charts, confusion matrices, and MLP loss curves
5. Saves `artifacts/experiment_logs/model_comparison.csv`

> **Swap the synthetic data block** with your real `engineer_features(...)` output when data is available.

In [ ]:
import sys, os
from pathlib import Path

# Make project root importable
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent          # …/pathsnap/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless – switch to 'TkAgg' if interactive
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, log_loss,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

# Output directories
PLOTS_DIR = PROJECT_ROOT / 'results' / 'plots'
LOGS_DIR  = PROJECT_ROOT / 'artifacts' / 'experiment_logs'
MODELS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
SCALERS_DIR = PROJECT_ROOT / 'artifacts' / 'scalers'
for _d in [PLOTS_DIR, LOGS_DIR, MODELS_DIR, SCALERS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print('Directories OK')

## 1. Load Features

Replace the synthetic block below with real data from `feature_engineering.engineer_features()`.

In [ ]:
# ---------------------------------------------------------------------------
# Option A — load real data (uncomment when data pipeline is ready)
# ---------------------------------------------------------------------------
# from src.preprocessing.data_loader import load_partition
# from src.features.feature_engineering import engineer_features
# import numpy as np
#
# trips = load_partition(PROJECT_ROOT / 'data' / 'train')
# Xs, ys = [], []
# for trip in trips:
#     X_trip, y_trip = engineer_features(trip)
#     Xs.append(X_trip); ys.append(y_trip)
# X_all = np.vstack(Xs)
# y_all = np.concatenate(ys)

# ---------------------------------------------------------------------------
# Option B — synthetic data (stand-alone demo)
# ---------------------------------------------------------------------------
from sklearn.datasets import make_classification
import numpy as np

FEATURE_NAMES = [
    'perp_dist_m', 'proj_ratio', 'bearing_diff_deg', 'speed_mps',
    'time_gap_s', 'from_node_lon', 'from_node_lat',
    'to_node_lon', 'to_node_lat', 'arc_length_m'
]

X_all, y_all = make_classification(
    n_samples=8000,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    weights=[0.80, 0.20],   # ~4:1 imbalance typical for k=5 candidates
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

print(f'Dataset shape : {X_all.shape}')
print(f'Class counts  : {np.bincount(y_all)}  (ratio {np.bincount(y_all)[0]/np.bincount(y_all)[1]:.1f}:1)')

In [ ]:
# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train class dist: {np.bincount(y_train)}')
print(f'Test  class dist: {np.bincount(y_test)}')

## 2. Preprocessing — Scaling & Imbalance Handling

In [ ]:
# --- StandardScaler ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

scaler_path = SCALERS_DIR / 'scaler.joblib'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved → {scaler_path}')

# --- Imbalance check ---
counts = np.bincount(y_train)
imbalance_ratio = counts.max() / counts.min()
print(f'Imbalance ratio: {imbalance_ratio:.2f}:1')

if imbalance_ratio > 5:
    print('Applying SMOTE …')
    try:
        from imblearn.over_sampling import SMOTE
        sm = SMOTE(random_state=RANDOM_STATE)
        X_train_cv, y_train_cv = sm.fit_resample(X_train_s, y_train)
        print(f'After SMOTE: {np.bincount(y_train_cv)}')
    except ImportError:
        print('imbalanced-learn not installed – using class_weight instead')
        X_train_cv, y_train_cv = X_train_s, y_train
else:
    print('Imbalance within acceptable range – no SMOTE needed')
    X_train_cv, y_train_cv = X_train_s, y_train

## 3. Define Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, solver='lbfgs',
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Naive Bayes': GaussianNB(),
    'K-Nearest Neighbours': KNeighborsClassifier(
        n_neighbors=7, weights='distance', n_jobs=-1
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_leaf=3,
        max_features='sqrt', class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        subsample=0.8, random_state=RANDOM_STATE
    ),
    'MLP Neural Network': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
        alpha=1e-4, learning_rate='adaptive', max_iter=300,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
        random_state=RANDOM_STATE
    ),
}

print(f'{len(MODELS)} models defined')

## 4. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in MODELS.items():
    print(f'  Cross-validating: {name} …', end=' ', flush=True)
    t0 = time.perf_counter()
    scores = cross_validate(
        model, X_train_cv, y_train_cv,
        cv=skf,
        scoring=['f1_macro', 'accuracy', 'neg_log_loss'],
        n_jobs=1,
        return_train_score=False,
    )
    elapsed = time.perf_counter() - t0
    cv_rows.append({
        'Model':           name,
        'CV Macro F1':     scores['test_f1_macro'].mean(),
        'CV F1 Std':       scores['test_f1_macro'].std(),
        'CV Accuracy':     scores['test_accuracy'].mean(),
        'CV Acc Std':      scores['test_accuracy'].std(),
        'CV Log Loss':     -scores['test_neg_log_loss'].mean(),
        'CV Time (s)':     elapsed,
    })
    print(f"macro-F1 {scores['test_f1_macro'].mean():.4f} ± {scores['test_f1_macro'].std():.4f}")

cv_df = pd.DataFrame(cv_rows).sort_values('CV Macro F1', ascending=False).reset_index(drop=True)
cv_df

## 5. Test-Set Evaluation (refit on full training set)

In [ ]:
test_rows = []
fitted_models = {}

for name, model in MODELS.items():
    model.fit(X_train_cv, y_train_cv)
    fitted_models[name] = model

    t0 = time.perf_counter()
    y_pred = model.predict(X_test_s)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_s)
    else:
        y_prob = np.eye(2)[y_pred]
    inf_time = time.perf_counter() - t0

    test_rows.append({
        'Model':             name,
        'Test Accuracy':     accuracy_score(y_test, y_pred),
        'Test Macro F1':     f1_score(y_test, y_pred, average='macro', zero_division=0),
        'Test Log Loss':     log_loss(y_test, y_prob),
        'Inference Time(s)': inf_time,
    })

test_df = pd.DataFrame(test_rows).sort_values('Test Macro F1', ascending=False).reset_index(drop=True)
test_df

In [ ]:
# Merge CV + test results
comparison_df = cv_df.merge(test_df, on='Model')
comparison_df = comparison_df.sort_values('CV Macro F1', ascending=False).reset_index(drop=True)

csv_path = LOGS_DIR / 'model_comparison.csv'
comparison_df.to_csv(csv_path, index=False)
print(f'Saved → {csv_path}')
comparison_df

## 6. Visualisations

### 6a. Bar Chart — CV Macro F1

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
models_ordered = comparison_df['Model'].tolist()
f1_means = comparison_df['CV Macro F1'].tolist()
f1_stds  = comparison_df['CV F1 Std'].tolist()

bars = ax.barh(models_ordered, f1_means, xerr=f1_stds,
               color=sns.color_palette('muted', len(models_ordered)),
               capsize=4, edgecolor='black', linewidth=0.6)
ax.set_xlabel('CV Macro F1 (mean ± std)', fontsize=12)
ax.set_title('5-Fold Cross-Validation Macro F1 by Model', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1)
for bar, v in zip(bars, f1_means):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cv_macro_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cv_macro_f1_comparison.png')

### 6b. Bar Chart — Accuracy, Log Loss, Inference Time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('CV Accuracy', 'CV Accuracy (mean)', axes[0]),
    ('CV Log Loss', 'CV Log Loss (mean)', axes[1]),
    ('Inference Time(s)', 'Inference Time (s)', axes[2]),
]

for col, title, ax in metrics:
    vals = comparison_df.set_index('Model')[col]
    vals.plot(kind='barh', ax=ax, color=sns.color_palette('pastel', len(vals)))
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.invert_yaxis()

plt.suptitle('Model Comparison — Secondary Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'model_secondary_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_secondary_metrics.png')

### 6c. Confusion Matrices — Top 3 Models

In [ ]:
top3 = comparison_df['Model'].head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, model_name in zip(axes, top3):
    model = fitted_models[model_name]
    y_pred = model.predict(X_test_s)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Wrong Arc (0)', 'Correct Arc (1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(model_name, fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices — Top 3 Models', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'confusion_matrices_top3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrices_top3.png')

### 6d. MLP — Training vs Validation Loss Curve

In [ ]:
mlp_model = fitted_models['MLP Neural Network']

# Re-fit with verbose to capture loss curve
mlp_track = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32), activation='relu', solver='adam',
    alpha=1e-4, learning_rate='adaptive', max_iter=300,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
    random_state=RANDOM_STATE, verbose=False,
)
mlp_track.fit(X_train_cv, y_train_cv)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mlp_track.loss_curve_, label='Training Loss', color='steelblue', lw=2)
if hasattr(mlp_track, 'validation_scores_') and mlp_track.validation_scores_ is not None:
    # validation_scores_ is accuracy; convert to 1-accuracy as proxy
    val_loss_proxy = [1 - s for s in mlp_track.validation_scores_]
    ax.plot(val_loss_proxy, label='Validation Loss (1−acc proxy)',
            color='tomato', lw=2, linestyle='--')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('MLP Neural Network — Training vs Validation Loss', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'mlp_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved mlp_loss_curve.png')

## 7. Classification Reports — Top 3 Models

In [ ]:
for model_name in top3:
    model = fitted_models[model_name]
    y_pred = model.predict(X_test_s)
    print(f'\n=== {model_name} ===')
    print(classification_report(y_test, y_pred,
                                 target_names=['Wrong Arc', 'Correct Arc'],
                                 zero_division=0))

## 8. Feature Importance (Random Forest)

In [ ]:
rf_model = fitted_models.get('Random Forest')
if rf_model is not None and hasattr(rf_model, 'feature_importances_'):
    imp = rf_model.feature_importances_
    idx = np.argsort(imp)[::-1]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(range(len(imp)), imp[idx],
           color=sns.color_palette('Blues_r', len(imp)))
    ax.set_xticks(range(len(imp)))
    ax.set_xticklabels(
        [FEATURE_NAMES[i] if i < len(FEATURE_NAMES) else f'feat_{i}' for i in idx],
        rotation=35, ha='right', fontsize=9
    )
    ax.set_ylabel('Importance', fontsize=11)
    ax.set_title('Random Forest Feature Importances', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved rf_feature_importance.png')
else:
    print('Random Forest not in fitted models or lacks feature_importances_')

## 9. Summary

The best-performing model (by CV macro-F1) will be used for Bayesian HPO in notebook `03_Bayesian_HPO.ipynb`.

In [ ]:
best_row = comparison_df.iloc[0]
print('=' * 55)
print(f'BEST MODEL : {best_row["Model"]}')
print(f'CV Macro F1: {best_row["CV Macro F1"]:.4f} ± {best_row["CV F1 Std"]:.4f}')
print(f'CV Accuracy: {best_row["CV Accuracy"]:.4f}')
print(f'CV Log Loss: {best_row["CV Log Loss"]:.4f}')
print('=' * 55)
print(f'\n→ Proceed to 03_Bayesian_HPO.ipynb to tune: {best_row["Model"]}')

# Save best model name for downstream notebook
import json
meta = {
    'best_model':      best_row['Model'],
    'cv_macro_f1':     round(float(best_row['CV Macro F1']), 6),
    'cv_f1_std':       round(float(best_row['CV F1 Std']),   6),
    'test_macro_f1':   round(float(best_row['Test Macro F1']), 6),
}
with open(LOGS_DIR / 'best_model_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print('Metadata saved → artifacts/experiment_logs/best_model_meta.json')